In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


import tensorflow as tf

from tensorflow.keras.utils import image_dataset_from_directory

# Initialize rng
rng = np.random.default_rng(2023)

In [ ]:
batch_size = 32 # This is a tunable hyperparameter
shape = (64, 64) # note we are reducing the size of the image

data_dir = r'/content/drive/MyDrive/UCMerced_LandUse/split'

train_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(data_dir, 'train'),
                                                       seed=rng.integers(500000),
                                                       image_size=shape,
                                                       batch_size=batch_size,
                                                       label_mode='categorical')

val_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(data_dir, 'val'),
                                                     seed=rng.integers(500000),
                                                     image_size=shape,
                                                     batch_size=batch_size,
                                                     label_mode='categorical')

test_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(data_dir, 'test'),
                                                      seed=rng.integers(500000),
                                                      image_size=shape,
                                                      batch_size=batch_size,
                                                      label_mode='categorical')

In [ ]:
# Visualize some images
fig, ax = plt.subplots(3,7, figsize=(20, 10))

train_dir = os.path.join(data_dir, 'train')
i = 0
j = 0
for class_name in [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]:
  im = plt.imread(os.path.join(train_dir, class_name, os.listdir(os.path.join(train_dir, class_name))[0]))
  ax[i][j].imshow(im);
  ax[i][j].set_title(class_name);

  j += 1
  if j>6:
    j = 0
    i += 1

# Start your solution below

In [ ]:
!pip install bayesian-optimization

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from sklearn.metrics import roc_curve, auc
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.activations import relu, sigmoid, tanh
from tensorflow.keras.metrics import AUC
from sklearn.model_selection import RandomizedSearchCV
from bayes_opt import BayesianOptimization
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
# Convert TensorFlow dataset to NumPy arrays
train_data = []
train_labels = []

for batch_data, batch_labels in train_ds:
    train_data.append(batch_data)
    train_labels.append(batch_labels)

train_data = np.vstack(train_data)
train_labels = np.vstack(train_labels)

In [ ]:
train_data.shape

In [ ]:
# Convert TensorFlow dataset to NumPy arrays
test_data = []
test_labels = []

for batch_data, batch_labels in test_ds:
    test_data.append(batch_data)
    test_labels.append(batch_labels)

test_data = np.vstack(test_data)
test_labels = np.vstack(test_labels)

In [ ]:
test_data.shape

In [ ]:
# Convert TensorFlow dataset to NumPy arrays
val_data = []
val_labels = []

for batch_data, batch_labels in val_ds:
    val_data.append(batch_data)
    val_labels.append(batch_labels)

val_data = np.vstack(val_data)
val_labels = np.vstack(val_labels)

In [ ]:
val_data.shape

# MLP with One Hidden

In [ ]:
# Define a function to create an MLP model with one hidden layer
def create_mlp_one_hidden(neurons, learning_rate):
    model = keras.Sequential()

    # Add a convolutional layer with 32 filters, 3x3 kernel, and 'relu' activation
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(64, 64, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))  # Add max-pooling

    # Flatten the output of the convolutional layer
    model.add(Flatten())

    # Add the first hidden layer
    model.add(Dense(neurons, activation='relu'))

    # Output layer
    model.add(Dense(21, activation='softmax'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Define hyperparameter search space for MLP-1HL
pbounds = {
    'neurons': (16, 256),         # Lower and upper bounds for 'neurons'
    'learning_rate': (0.001, 0.01)  # Lower and upper bounds for 'learning_rate'
}


# Create a function to evaluate the model with given hyperparameters
def evaluate_mlp_one_hidden(neurons, learning_rate):
    model = create_mlp_one_hidden(neurons, learning_rate)
    model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)
    y_pred = np.argmax(model.predict(val_data), axis=1)
    val_accuracy = accuracy_score(np.argmax(val_labels, axis=1), y_pred)
    return val_accuracy

# Initialize Bayesian optimization
optimizer = BayesianOptimization(
    f=evaluate_mlp_one_hidden,
    pbounds=pbounds,
    verbose=2,  # Adjust verbosity level as needed
    random_state=1,  # Set a random seed for reproducibility
)

# Perform Bayesian optimization
optimizer.maximize(init_points=5, n_iter=10)

# Get the best hyperparameters
best_hyperparameters = optimizer.max['params']

# Train the best model with the selected hyperparameters
best_model = create_mlp_one_hidden(best_hyperparameters['neurons'], best_hyperparameters['learning_rate'])
best_model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)

# Evaluate the best model on the test dataset
test_accuracy = best_model.evaluate(test_data, test_labels, verbose=0)[1]

print("Best Hyperparameters for MLP-1HL:", best_hyperparameters)
print("Test Accuracy for MLP-1HL:", test_accuracy)

In [ ]:
# List of activation functions to try
activation_functions = ['relu', 'sigmoid', 'tanh']


# Initialize lists to store all true positives and false positives for each class
all_true_positives = {activation: [] for activation in activation_functions}
all_false_positives = {activation: [] for activation in activation_functions}
trained_models = {}  # Store trained models
results = {}
histories = {}  # Store training histories

# Evaluate models and calculate ROC curves for each class
for activation in activation_functions:
    print(f"Training with Activation Function: {activation}")

    # Create the model with the specified activation function
    model = create_mlp_one_hidden(best_hyperparameters['neurons'], best_hyperparameters['learning_rate'])

    # Train the model on the training dataset
    history = model.fit(train_data, train_labels, epochs=20, verbose=1)
    histories[activation] = history  # Store training history

    # Evaluate the model on the test dataset
    test_loss, test_accuracy = model.evaluate(test_data, test_labels, verbose=0)

    # Calculate ROC curves for each class (one-vs-all)
    y_pred = model.predict(test_data)
    n_classes = 21  # Number of classes

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

        # Store true positives and false positives for each class and activation
        all_true_positives[activation].append(tpr[i])
        all_false_positives[activation].append(fpr[i])

    # Store the trained model
    trained_models[activation] = model

    # Store the results
    results[activation] = {
        'test_accuracy': test_accuracy
    }

    # Print the number of epochs completed
    print(f"Training completed for Activation Function: {activation}, Epochs: {len(history.history['accuracy'])}\n")

# Display the results
for activation, result in results.items():
    print(f"Activation Function: {activation}")
    print(f"Test Accuracy: {result['test_accuracy']:.4f}\n")

Activation Function: relu

Training Time: The relu activation function exhibited efficient training, completing all 20 epochs without any issues.

Test Accuracy: The test accuracy with relu improved significantly to 0.2254, which is a substantial improvement compared to the previous results. This suggests that the model learned better representations of the data.

Activation Function: sigmoid

Training Time: Sigmoid showed a moderate training time, similar to the previous training.

Test Accuracy: The test accuracy for sigmoid remained at 0.1524, which is relatively low compared to relu. This suggests that relu is a more suitable activation function for this problem.

Activation Function: tanh

Training Time: Tanh resulted in a similar training time as before.

Test Accuracy: The test accuracy for tanh also remained relatively low at 0.1492, similar to sigmoid.

In [ ]:
# Plot training and validation loss curves and accuracy curves
for activation, history in histories.items():
    plt.figure(figsize=(12, 4))

    # Plot loss curves
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.title(f'{activation} Activation - Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot accuracy curves
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.title(f'{activation} Activation - Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize dictionaries to store ROC curve data for each activation function
roc_curves = {activation: {} for activation in activation_functions}

# Calculate ROC curves for each class (one-vs-all) for each activation function
for activation in activation_functions:
    y_pred = trained_models[activation].predict(test_data)  # Use the correct model
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc = auc(fpr, tpr)

        roc_curves[activation][i] = {
            'fpr': fpr,
            'tpr': tpr,
            'roc_auc': roc_auc
        }

# Plot ROC curves and AUC values for each activation function separately
for activation in activation_functions:
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1)
    for i in range(n_classes):
        plt.plot(
            roc_curves[activation][i]['fpr'],
            roc_curves[activation][i]['tpr'],
            lw=2,
            label=f'Class {i} ({activation} AUC = {roc_curves[activation][i]["roc_auc"]:.2f})'
        )
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Multiclass ROC Curves for Activation: {activation}')
    #plt.legend(loc='lower right')

    plt.subplot(2, 1, 2)
    auc_values = [roc_curves[activation][i]['roc_auc'] for i in range(n_classes)]
    class_labels = [f'Class {i}' for i in range(n_classes)]
    plt.bar(class_labels, auc_values, alpha=0.75, label=activation)
    plt.xlabel('Class')
    plt.ylabel('AUC')
    plt.title(f'AUC Values for Activation: {activation}')

    plt.tight_layout()

    plt.show()

# MLP with Two Hidden

In [ ]:
# Define a function to create an MLP model with one hidden layer
def create_mlp_two_hidden(neurons, learning_rate):
    model = keras.Sequential()

    # Add a convolutional layer with 32 filters, 3x3 kernel, and 'relu' activation
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(64, 64, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))  # Add max-pooling

    # Flatten the output of the convolutional layer
    model.add(Flatten())

    # Add the first hidden layer
    model.add(Dense(neurons, activation='relu'))

    # Output layer
    model.add(Dense(21, activation='softmax'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Define hyperparameter search space for MLP-1HL
pbounds = {
    'neurons': (16, 256),         # Lower and upper bounds for 'neurons'
    'learning_rate': (0.001, 0.01)  # Lower and upper bounds for 'learning_rate'
}

# Create a function to evaluate the model with given hyperparameters
def evaluate_mlp_two_hidden(neurons, learning_rate):
    model = create_mlp_two_hidden(neurons, learning_rate)
    model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)
    y_pred = np.argmax(model.predict(val_data), axis=1)
    val_accuracy = accuracy_score(np.argmax(val_labels, axis=1), y_pred)
    return val_accuracy

# Initialize Bayesian optimization
optimizer = BayesianOptimization(
    f=evaluate_mlp_two_hidden,
    pbounds=pbounds,
    verbose=2,  # Adjust verbosity level as needed
    random_state=1,  # Set a random seed for reproducibility
)

# Perform Bayesian optimization
optimizer.maximize(init_points=5, n_iter=10)

# Get the best hyperparameters
best_hyperparameters = optimizer.max['params']

# Train the best model with the selected hyperparameters
best_model = create_mlp_one_hidden(best_hyperparameters['neurons'], best_hyperparameters['learning_rate'])
best_model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)

# Evaluate the best model on the test dataset
test_accuracy = best_model.evaluate(test_data, test_labels, verbose=0)[1]

print("Best Hyperparameters for MLP-2HL:", best_hyperparameters)
print("Test Accuracy for MLP-2HL:", test_accuracy)

In [ ]:
# List of activation functions to try
activation_functions = ['relu', 'sigmoid', 'tanh']

# Initialize lists to store all true positives and false positives for each class
all_true_positives = {activation: [] for activation in activation_functions}
all_false_positives = {activation: [] for activation in activation_functions}
trained_models = {}  # Store trained models
results = {}
histories = {}  # Store training histories


# Evaluate models and calculate ROC curves for each class
for activation in activation_functions:
    print(f"Training with Activation Function: {activation}")

    # Create the model with the specified activation function
    model = create_mlp_one_hidden(best_hyperparameters['neurons'], best_hyperparameters['learning_rate'])

    # Train the model on the training dataset
    history = model.fit(train_data, train_labels, epochs=20, verbose=1)
    histories[activation] = history  # Store training history

    # Evaluate the model on the test dataset
    test_loss, test_accuracy = model.evaluate(test_data, test_labels, verbose=0)

    # Calculate ROC curves for each class (one-vs-all)
    y_pred = model.predict(test_data)
    n_classes = 21  # Number of classes

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

        # Store true positives and false positives for each class and activation
        all_true_positives[activation].append(tpr[i])
        all_false_positives[activation].append(fpr[i])

    # Store the trained model
    trained_models[activation] = model

    # Store the results
    results[activation] = {
        'test_accuracy': test_accuracy
    }

    # Print the number of epochs completed
    print(f"Training completed for Activation Function: {activation}, Epochs: {len(history.history['accuracy'])}\n")

# Display the results
for activation, result in results.items():
    print(f"Activation Function: {activation}")
    print(f"Test Accuracy: {result['test_accuracy']:.4f}\n")

Activation Function: relu

Training Time: The relu activation function continues to exhibit efficient training, completing all 20 epochs without any issues.

Test Accuracy: The test accuracy with relu improved even further to 0.2540, which is a significant improvement compared to the previous results. This suggests that the model has learned better representations of the data.


Activation Function: sigmoid

Training Time: Sigmoid shows a similar moderate training time as in the previous training.

Test Accuracy: Unfortunately, the test accuracy for sigmoid remains low at 0.0730, which is significantly lower than relu.

Activation Function: tanh

Training Time: Tanh still results in a similar training time as before.

Test Accuracy: The test accuracy for tanh also remains relatively low at 0.1397, similar to sigmoid.

In [ ]:
# Plot training and validation loss curves and accuracy curves
for activation, history in histories.items():
    plt.figure(figsize=(12, 4))

    # Plot loss curves
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.title(f'{activation} Activation - Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot accuracy curves
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.title(f'{activation} Activation - Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize dictionaries to store ROC curve data for each activation function
roc_curves = {activation: {} for activation in activation_functions}

# Calculate ROC curves for each class (one-vs-all) for each activation function
for activation in activation_functions:
    y_pred = trained_models[activation].predict(test_data)  # Use the correct model
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc = auc(fpr, tpr)

        roc_curves[activation][i] = {
            'fpr': fpr,
            'tpr': tpr,
            'roc_auc': roc_auc
        }

# Plot ROC curves and AUC values for each activation function separately
for activation in activation_functions:
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1)
    for i in range(n_classes):
        plt.plot(
            roc_curves[activation][i]['fpr'],
            roc_curves[activation][i]['tpr'],
            lw=2,
            label=f'Class {i} ({activation} AUC = {roc_curves[activation][i]["roc_auc"]:.2f})'
        )
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Multiclass ROC Curves for Activation: {activation}')
    #plt.legend(loc='lower right')

    plt.subplot(2, 1, 2)
    auc_values = [roc_curves[activation][i]['roc_auc'] for i in range(n_classes)]
    class_labels = [f'Class {i}' for i in range(n_classes)]
    plt.bar(class_labels, auc_values, alpha=0.75, label=activation)
    plt.xlabel('Class')
    plt.ylabel('AUC')
    plt.title(f'AUC Values for Activation: {activation}')

    plt.tight_layout()

    plt.show()

# MLP with two hidden layers and dropout after each of the two hidden layers.


In [ ]:
def create_mlp_two_hidden_dropout(neurons, dropout_rate, learning_rate):
    model = keras.Sequential()

    # Add a convolutional layer with 32 filters, 3x3 kernel, and 'relu' activation
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(64, 64, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))  # Add max-pooling

    # Flatten the output of the convolutional layer
    model.add(Flatten())

    # Add the first hidden layer with dropout
    model.add(Dense(neurons, activation='relu'))
    model.add(Dropout(dropout_rate))  # Dropout layer

    # Add the second hidden layer with dropout
    model.add(Dense(neurons, activation='relu'))
    model.add(Dropout(dropout_rate))  # Dropout layer

    # Output layer
    model.add(Dense(21, activation='softmax'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Define hyperparameter search space for MLP-1HL
pbounds = {
    'neurons': (16, 256),         # Lower and upper bounds for 'neurons'
    'learning_rate': (0.001, 0.01),  # Lower and upper bounds for 'learning_rate'
    'dropout_rate': (0.2, 0.6)
}

# Create a function to evaluate the model with given hyperparameters
def evaluate_mlp_two_hidden_dropout(neurons, learning_rate, dropout_rate):
    model = create_mlp_two_hidden_dropout(neurons, learning_rate, dropout_rate)
    model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)
    y_pred = np.argmax(model.predict(val_data), axis=1)
    val_accuracy = accuracy_score(np.argmax(val_labels, axis=1), y_pred)
    return val_accuracy

# Initialize Bayesian optimization
optimizer = BayesianOptimization(
    f=evaluate_mlp_two_hidden_dropout,
    pbounds=pbounds,
    verbose=2,  # Adjust verbosity level as needed
    random_state=1,  # Set a random seed for reproducibility
)

# Perform Bayesian optimization
optimizer.maximize(init_points=5, n_iter=10)

# Get the best hyperparameters
best_hyperparameters = optimizer.max['params']

# Train the best model with the selected hyperparameters
best_model = create_mlp_two_hidden_dropout(best_hyperparameters['neurons'], best_hyperparameters['dropout_rate'] , best_hyperparameters['learning_rate'])
best_model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)

# Evaluate the best model on the test dataset
test_accuracy = best_model.evaluate(test_data, test_labels, verbose=0)[1]

print("Best Hyperparameters for MLP-2HL-Dropout:", best_hyperparameters)
print("Test Accuracy for MLP-2HL-Dropout:", test_accuracy)

In [ ]:
# List of activation functions to try
activation_functions = ['relu', 'sigmoid', 'tanh']

# Initialize lists to store all true positives and false positives for each class
all_true_positives = {activation: [] for activation in activation_functions}
all_false_positives = {activation: [] for activation in activation_functions}
trained_models = {}  # Store trained models
results = {}
histories = {}  # Store training histories

# Evaluate models and calculate ROC curves for each class
for activation in activation_functions:
    print(f"Training with Activation Function: {activation}")

    # Create the model with the specified activation function
    model = create_mlp_two_hidden_dropout(best_hyperparameters['neurons'], best_hyperparameters['dropout_rate'] , best_hyperparameters['learning_rate'])

    # Train the model on the training dataset
    history = model.fit(train_data, train_labels, epochs=20, verbose=1)
    histories[activation] = history  # Store training history

    # Evaluate the model on the test dataset
    test_loss, test_accuracy = model.evaluate(test_data, test_labels, verbose=0)

    # Calculate ROC curves for each class (one-vs-all)
    y_pred = model.predict(test_data)
    n_classes = 21  # Number of classes

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

        # Store true positives and false positives for each class and activation
        all_true_positives[activation].append(tpr[i])
        all_false_positives[activation].append(fpr[i])

    # Store the trained model
    trained_models[activation] = model

    # Store the results
    results[activation] = {
        'test_accuracy': test_accuracy
    }

    # Print the number of epochs completed
    print(f"Training completed for Activation Function: {activation}, Epochs: {len(history.history['accuracy'])}\n")

# Display the results
for activation, result in results.items():
    print(f"Activation Function: {activation}")
    print(f"Test Accuracy: {result['test_accuracy']:.4f}\n")

Activation Function: relu

Training Time: The training with the relu activation function took a reasonable amount of time, and all 20 epochs were completed.

Test Accuracy: Unfortunately, the test accuracy with relu has decreased to 0.0571, which is lower than the previous result. This suggests that the model's performance is not stable, and it might be encountering convergence issues.

Activation Function: sigmoid

Training Time: Sigmoid also took a reasonable amount of time to complete all 20 epochs.

Test Accuracy: The test accuracy with sigmoid has improved slightly to 0.0603, but it remains at a very low level. Sigmoid activation is still not performing well for this problem.

Activation Function: tanh

Training Time: Training with the tanh activation function also took a reasonable amount of time.

Test Accuracy: Tanh continues to show a low test accuracy of 0.0476, which is similar to the previous result.

In [ ]:
# Plot training and validation loss curves and accuracy curves
for activation, history in histories.items():
    plt.figure(figsize=(12, 4))

    # Plot loss curves
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.title(f'{activation} Activation - Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot accuracy curves
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.title(f'{activation} Activation - Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize dictionaries to store ROC curve data for each activation function
roc_curves = {activation: {} for activation in activation_functions}

# Calculate ROC curves for each class (one-vs-all) for each activation function
for activation in activation_functions:
    y_pred = trained_models[activation].predict(test_data)  # Use the correct model
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc = auc(fpr, tpr)

        roc_curves[activation][i] = {
            'fpr': fpr,
            'tpr': tpr,
            'roc_auc': roc_auc
        }

# Plot ROC curves and AUC values for each activation function separately
for activation in activation_functions:
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1)
    for i in range(n_classes):
        plt.plot(
            roc_curves[activation][i]['fpr'],
            roc_curves[activation][i]['tpr'],
            lw=2,
            label=f'Class {i} ({activation} AUC = {roc_curves[activation][i]["roc_auc"]:.2f})'
        )
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Multiclass ROC Curves for Activation: {activation}')
    #plt.legend(loc='lower right')

    plt.subplot(2, 1, 2)
    auc_values = [roc_curves[activation][i]['roc_auc'] for i in range(n_classes)]
    class_labels = [f'Class {i}' for i in range(n_classes)]
    plt.bar(class_labels, auc_values, alpha=0.75, label=activation)
    plt.xlabel('Class')
    plt.ylabel('AUC')
    plt.title(f'AUC Values for Activation: {activation}')

    plt.tight_layout()

    plt.show()

# MLP with two hidden layers and batch normalization and dropout after each of the two hidden layers.

In [ ]:
# Define a function to create an MLP model with two hidden layers, batch normalization, and dropout
def create_mlp_two_hidden_with_batchnorm_dropout(neurons, learning_rate, dropout_rate):
    model = keras.Sequential()
    # Add a convolutional layer with 32 filters, 3x3 kernel, and 'relu' activation
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(64, 64, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))  # Add max-pooling

    # Flatten the output of the convolutional layer
    model.add(Flatten())

    # Add the first hidden layer
    model.add(Dense(neurons, activation='relu'))
    model.add(BatchNormalization())  # Batch normalization after the first hidden layer
    model.add(Dropout(dropout_rate))  # Dropout after the first hidden layer

    # Add the second hidden layer
    model.add(Dense(neurons, activation='relu'))
    model.add(BatchNormalization())  # Batch normalization after the second hidden layer
    model.add(Dropout(dropout_rate))  # Dropout after the second hidden layer

    # Output layer with 21 classes (adjust as needed)
    model.add(Dense(21, activation='softmax'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

## Define hyperparameter search space for MLP-1HL
pbounds = {
    'neurons': (16, 256),         # Lower and upper bounds for 'neurons'
    'learning_rate': (0.001, 0.01),  # Lower and upper bounds for 'learning_rate'
    'dropout_rate': (0.2, 0.6)
}

# Create a function to evaluate the model with given hyperparameters
def evaluate_mlp_two_hidden_with_batchnorm_dropout(neurons, learning_rate, dropout_rate):
    model = create_mlp_two_hidden_with_batchnorm_dropout(neurons, learning_rate, dropout_rate)
    model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)
    y_pred = np.argmax(model.predict(val_data), axis=1)
    val_accuracy = accuracy_score(np.argmax(val_labels, axis=1), y_pred)
    return val_accuracy

# Initialize Bayesian optimization
optimizer = BayesianOptimization(
    f=evaluate_mlp_two_hidden_with_batchnorm_dropout,
    pbounds=pbounds,
    verbose=2,  # Adjust verbosity level as needed
    random_state=1,  # Set a random seed for reproducibility
)

# Perform Bayesian optimization
optimizer.maximize(init_points=5, n_iter=10)

# Get the best hyperparameters
best_hyperparameters = optimizer.max['params']

# Train the best model with the selected hyperparameters
best_model = create_mlp_two_hidden_with_batchnorm_dropout(best_hyperparameters['neurons'], best_hyperparameters['dropout_rate'] , best_hyperparameters['learning_rate'])
best_model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)

# Evaluate the best model on the test dataset
test_accuracy = best_model.evaluate(test_data, test_labels, verbose=0)[1]

print("Best Hyperparameters for MLP-2HL with Batch Norm and Dropout:", best_hyperparameters)
print("Test Accuracy for MLP-2HL with Batch Norm and Dropout:", test_accuracy)

In [ ]:
# Initialize lists to store all true positives and false positives for each class
all_true_positives = {activation: [] for activation in activation_functions}
all_false_positives = {activation: [] for activation in activation_functions}
trained_models = {}  # Store trained models
results = {}
histories = {}  # Store training histories


# Evaluate models and calculate ROC curves for each class
for activation in activation_functions:
    print(f"Training with Activation Function: {activation}")

    # Create the model with the specified activation function
    model = create_mlp_two_hidden_with_batchnorm_dropout(best_hyperparameters['neurons'], best_hyperparameters['dropout_rate'] , best_hyperparameters['learning_rate'])

    # Train the model on the training dataset
    history = model.fit(train_data, train_labels, epochs=20, verbose=1)
    histories[activation] = history  # Store training history

    # Evaluate the model on the test dataset
    test_loss, test_accuracy = model.evaluate(test_data, test_labels, verbose=0)

    # Calculate ROC curves for each class (one-vs-all)
    y_pred = model.predict(test_data)
    n_classes = 21  # Number of classes

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

        # Store true positives and false positives for each class and activation
        all_true_positives[activation].append(tpr[i])
        all_false_positives[activation].append(fpr[i])

    # Store the trained model
    trained_models[activation] = model

    # Store the results
    results[activation] = {
        'test_accuracy': test_accuracy
    }

    # Print the number of epochs completed
    print(f"Training completed for Activation Function: {activation}, Epochs: {len(history.history['accuracy'])}\n")

# Display the results
for activation, result in results.items():
    print(f"Activation Function: {activation}")
    print(f"Test Accuracy: {result['test_accuracy']:.4f}\n")

Activation Function: relu

Training Time: Training with the relu activation function took a reasonable amount of time, and all 20 epochs were completed.

Test Accuracy: The test accuracy with relu has improved significantly to 0.1524, which is a notable improvement compared to the previous result. This suggests that the model might be learning better representations of the data with more training.

Activation Function: sigmoid

Training Time: Sigmoid also took a reasonable amount of time to complete all 20 epochs.

Test Accuracy: The test accuracy with sigmoid has improved substantially to 0.1841. This is a notable improvement over the previous results and indicates that the model is learning better representations of the data with more training when using sigmoid activation.


Activation Function: tanh

Training Time: Training with the tanh activation function also took a reasonable amount of time.

Test Accuracy: Tanh activation shows a test accuracy of 0.1619, which is an improvement over the previous result but still relatively low.

In [ ]:
# Plot training and validation loss curves and accuracy curves
for activation, history in histories.items():
    plt.figure(figsize=(12, 4))

    # Plot loss curves
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.title(f'{activation} Activation - Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot accuracy curves
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.title(f'{activation} Activation - Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize dictionaries to store ROC curve data for each activation function
roc_curves = {activation: {} for activation in activation_functions}

# Calculate ROC curves for each class (one-vs-all) for each activation function
for activation in activation_functions:
    y_pred = trained_models[activation].predict(test_data)  # Use the correct model
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc = auc(fpr, tpr)

        roc_curves[activation][i] = {
            'fpr': fpr,
            'tpr': tpr,
            'roc_auc': roc_auc
        }

# Plot ROC curves and AUC values for each activation function separately
for activation in activation_functions:
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1)
    for i in range(n_classes):
        plt.plot(
            roc_curves[activation][i]['fpr'],
            roc_curves[activation][i]['tpr'],
            lw=2,
            label=f'Class {i} ({activation} AUC = {roc_curves[activation][i]["roc_auc"]:.2f})'
        )
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Multiclass ROC Curves for Activation: {activation}')
    #plt.legend(loc='lower right')

    plt.subplot(2, 1, 2)
    auc_values = [roc_curves[activation][i]['roc_auc'] for i in range(n_classes)]
    class_labels = [f'Class {i}' for i in range(n_classes)]
    plt.bar(class_labels, auc_values, alpha=0.75, label=activation)
    plt.xlabel('Class')
    plt.ylabel('AUC')
    plt.title(f'AUC Values for Activation: {activation}')

    plt.tight_layout()

    plt.show()

# Choice of MLP

In [ ]:
# Define a function to create a three-layer MLP model with Conv2D layers, batch normalization, and dropout
def create_mlp_three_layers_with_batchnorm_dropout(neurons, learning_rate, dropout_rate):
    model = Sequential()

    # Add the first Conv2D layer with 32 filters, 3x3 kernel, 'relu' activation, and input shape for images
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(64, 64, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))  # Add max-pooling

    # Optionally add batch normalization after the first Conv2D layer
    model.add(BatchNormalization())

    # Optionally add dropout after the first Conv2D layer
    model.add(Dropout(dropout_rate))

    # Add the second Conv2D layer with 64 filters, 3x3 kernel, and 'relu' activation
    model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))  # Add max-pooling

    # Optionally add batch normalization after the second Conv2D layer
    model.add(BatchNormalization())

    # Optionally add dropout after the second Conv2D layer
    model.add(Dropout(dropout_rate))

    # Flatten the output of the Conv2D layers
    model.add(Flatten())

    # Add the first dense hidden layer
    model.add(Dense(neurons, activation='relu'))

    # Optionally add batch normalization after the first dense hidden layer
    model.add(BatchNormalization())

    # Optionally add dropout after the first dense hidden layer
    model.add(Dropout(dropout_rate))

    # Add the second dense hidden layer
    model.add(Dense(neurons, activation='relu'))

    # Optionally add batch normalization after the second dense hidden layer
    model.add(BatchNormalization())

    # Optionally add dropout after the second dense hidden layer
    model.add(Dropout(dropout_rate))

    # Add the output layer with 21 classes (adjust as needed)
    model.add(Dense(21, activation='softmax'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

## Define hyperparameter search space for MLP-1HL
pbounds = {
    'neurons': (16, 256),         # Lower and upper bounds for 'neurons'
    'learning_rate': (0.001, 0.01),  # Lower and upper bounds for 'learning_rate'
    'dropout_rate': (0.2, 0.6)
}

# Create a function to evaluate the model with given hyperparameters
def evaluate_mlp_three_layers_with_batchnorm_dropout(neurons, learning_rate, dropout_rate):
    model = create_mlp_two_hidden_with_batchnorm_dropout(neurons, learning_rate, dropout_rate)
    model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)
    y_pred = np.argmax(model.predict(val_data), axis=1)
    val_accuracy = accuracy_score(np.argmax(val_labels, axis=1), y_pred)
    return val_accuracy

# Initialize Bayesian optimization
optimizer = BayesianOptimization(
    f=evaluate_mlp_three_layers_with_batchnorm_dropout,
    pbounds=pbounds,
    verbose=2,  # Adjust verbosity level as needed
    random_state=1,  # Set a random seed for reproducibility
)

# Perform Bayesian optimization
optimizer.maximize(init_points=5, n_iter=10)

# Get the best hyperparameters
best_hyperparameters = optimizer.max['params']

# Train the best model with the selected hyperparameters
best_model = create_mlp_three_layers_with_batchnorm_dropout(best_hyperparameters['neurons'], best_hyperparameters['dropout_rate'] , best_hyperparameters['learning_rate'])
best_model.fit(train_data, train_labels, epochs=20, batch_size=32, verbose=0)

# Evaluate the best model on the test dataset
test_accuracy = best_model.evaluate(test_data, test_labels, verbose=0)[1]

print("Best Hyperparameters for Three-Layer MLP with Batch Norm and Dropout:", best_hyperparameters)
print("Test Accuracy for Three-Layer MLP with Batch Norm and Dropout:", test_accuracy)

In [ ]:
# Initialize lists to store all true positives and false positives for each class
all_true_positives = {activation: [] for activation in activation_functions}
all_false_positives = {activation: [] for activation in activation_functions}
trained_models = {}  # Store trained models
results = {}
histories = {}  # Store training histories

# List of activation functions to try
activation_functions = ['relu', 'sigmoid', 'tanh']

# Evaluate models and calculate ROC curves for each class
for activation in activation_functions:
    print(f"Training with Activation Function: {activation}")

    # Create the model with the specified activation function
    model =  create_mlp_three_layers_with_batchnorm_dropout(best_hyperparameters['neurons'], best_hyperparameters['dropout_rate'] , best_hyperparameters['learning_rate'])

    # Train the model on the training dataset
    history = model.fit(train_data, train_labels, epochs=20, verbose=1)
    histories[activation] = history  # Store training history

    # Evaluate the model on the test dataset
    test_loss, test_accuracy = model.evaluate(test_data, test_labels, verbose=0)

    # Calculate ROC curves for each class (one-vs-all)
    y_pred = model.predict(test_data)
    n_classes = 21  # Number of classes

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

        # Store true positives and false positives for each class and activation
        all_true_positives[activation].append(tpr[i])
        all_false_positives[activation].append(fpr[i])

    # Store the trained model
    trained_models[activation] = model

    # Store the results
    results[activation] = {
        'test_accuracy': test_accuracy
    }

    # Print the number of epochs completed
    print(f"Training completed for Activation Function: {activation}, Epochs: {len(history.history['accuracy'])}\n")

# Display the results
for activation, result in results.items():
    print(f"Activation Function: {activation}")
    print(f"Test Accuracy: {result['test_accuracy']:.4f}\n")

Activation Function: relu

Training Time: Training with the relu activation function took a reasonable amount of time, and all 20 epochs were completed.

Test Accuracy: The test accuracy with relu has improved to 0.2063, which is a notable improvement compared to the previous result. This suggests that the model might be learning better representations of the data with more training.


Activation Function: sigmoid

Training Time: Sigmoid also took a reasonable amount of time to complete all 20 epochs.

Test Accuracy: The test accuracy with sigmoid has significantly decreased to 0.0254. This is a significant drop compared to the previous result. It appears that the model may not be learning effectively with the sigmoid activation function in this case.

Activation Function: tanh

Training Time: Training with the tanh activation function also took a reasonable amount of time.

Test Accuracy: Tanh activation shows the highest test accuracy of 0.4063. This is a significant improvement over the previous results and indicates that the model is learning better representations of the data with more training when using tanh activation.

In [ ]:
# Plot training and validation loss curves and accuracy curves
for activation, history in histories.items():
    plt.figure(figsize=(12, 4))

    # Plot loss curves
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.title(f'{activation} Activation - Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot accuracy curves
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.title(f'{activation} Activation - Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize dictionaries to store ROC curve data for each activation function
roc_curves = {activation: {} for activation in activation_functions}

# Calculate ROC curves for each class (one-vs-all) for each activation function
for activation in activation_functions:
    y_pred = trained_models[activation].predict(test_data)  # Use the correct model
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(test_labels[:, i], y_pred[:, i])
        roc_auc = auc(fpr, tpr)

        roc_curves[activation][i] = {
            'fpr': fpr,
            'tpr': tpr,
            'roc_auc': roc_auc
        }

# Plot ROC curves and AUC values for each activation function separately
for activation in activation_functions:
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1)
    for i in range(n_classes):
        plt.plot(
            roc_curves[activation][i]['fpr'],
            roc_curves[activation][i]['tpr'],
            lw=2,
            label=f'Class {i} ({activation} AUC = {roc_curves[activation][i]["roc_auc"]:.2f})'
        )
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Multiclass ROC Curves for Activation: {activation}')
    #plt.legend(loc='lower right')

    plt.subplot(2, 1, 2)
    auc_values = [roc_curves[activation][i]['roc_auc'] for i in range(n_classes)]
    class_labels = [f'Class {i}' for i in range(n_classes)]
    plt.bar(class_labels, auc_values, alpha=0.75, label=activation)
    plt.xlabel('Class')
    plt.ylabel('AUC')
    plt.title(f'AUC Values for Activation: {activation}')

    plt.tight_layout()

    plt.show()

# Conclusion

We tested different configurations of neural networks to see which one works best for a particular task. The goal is to classify data accurately. Here's what we found:

One Hidden Layer: We started with a simple setup where the neural network had one layer in the middle. We tried different ways of processing the data:

Using "relu" activation, we got an accuracy of about 22.5%.
With "sigmoid" activation, it was around 15%.
And "tanh" activation gave us about 14.9%.
These results weren't very impressive, meaning our model struggled to understand the data patterns.

Two Hidden Layers: We then made the network a bit more complex by adding another layer:

Using "relu," we got an accuracy of around 25.4%. It improved a bit, but not significantly.
"Sigmoid" activation only got us 7.3%, which wasn't good.
"Tanh" activation performed better at 13.9%, but it wasn't great either.
So, adding a second layer helped somewhat, but not by a lot.

Two Hidden Layers with Dropout: We tried to prevent the model from memorizing the data too much by adding a dropout layer. Unfortunately, it didn't work well:

With "relu," accuracy dropped to just 5.7%.
"Sigmoid" got us 6.0%.
And "tanh" was at 4.8%.
The dropout layers made things worse, and the model didn't learn effectively.

Two Hidden Layers with Batch Normalization and Dropout: We then tried a different technique called batch normalization along with dropout:

"Relu" activation achieved 15.2%.
"Sigmoid" was slightly better at 18.4%.
"Tanh" reached 16.2%.
These models were more stable and performed better than those without batch normalization and dropout, but still not great.

Our Choice: MLP with Two Hidden Layers and tanh Activation: Finally, we tested a model with two hidden layers, batch normalization, and "tanh" activation:

This one performed the best with an accuracy of 40.6%.
Comparison and Insights:

The best model (MLP with two hidden layers, batch normalization, and tanh activation) achieved the highest accuracy without severe underfitting or overfitting.
Dropout and batch normalization helped prevent overfitting, but too much dropout hurt the model's performance.
To improve further, we can fine-tune the model's settings, try more complex architectures, experiment with different techniques, and consider regularization methods.
In simple terms, our best model did pretty well, but there's always room for improvement. It's like finding the right recipe – we need to tweak it a bit to make it even better.